# 月營收方向衝突與三月彙總品質檢查

此 notebook 用 `crawler.source_records` 的官方 TWSE/TPEx 月營收資料檢查期別完整性、上市/上櫃覆蓋、單月 YoY 與 YTD YoY 的方向衝突，並比較 2026/06–08 三個連續月份的合計 YoY。預期 grain 為一個交易所來源、證券代碼、營收月份一列。原始金額單位為新台幣千元。

## 執行方式與限制

- 在 DataFlow 的 Airflow scheduler Python 環境執行，`/opt/airflow/dags` 必須在 import path；連線由專案既有 `_open_connection()` 讀取設定，本 notebook 不列印憑證。
- 所有查詢皆為 `SELECT`。不要從 notebook 觸發 DAG、寫入 source catalog、通知或交易。
- 3M YoY 是三個連續月份營收總和對去年同三個月的比率，只作上下文，不加入現行方向票。益得去年同期含公司說明的一次性技術服務收入，故彙總率仍可能受比較基期扭曲。
- 資料列會被同一 grain 的後續來源更新覆寫；本分析使用目前資料庫可見版本，無法還原 2026/10/02 日線候選執行時的舊 payload。

## 2026/10/03 台北時間 04:04 的已觀察結果

| 市場 | 2026/06 rows | 2026/07 rows | 2026/08 rows | 完整三月代碼 | 單月 vs YTD 反向（06/07/08） | 單月 vs 3M 反向 | YTD vs 3M 反向 |
|---|---:|---:|---:|---:|---:|---:|---:|
| TWSE 上市 | 1082 | 1085 | 1086 | 1076 | 20.89% / 21.46% / 20.76% | 145/1076 (13.48%) | 165/1076 (15.33%) |
| TPEx 上櫃 | 891 | 890 | 892 | 887 | 22.44% / 19.93% / 22.21% | 126/887 (14.21%) | 137/887 (15.45%) |

1,963/1,976 個跨期出現的代碼有完整 6–8 月紀錄；有 13 個代碼缺至少一期。Oct 2 daily 候選中有 44/50 個可計算三月彙總，4 個單月與 3M 方向相反，3 個 YTD 與 3M 方向相反。這些是描述統計，不能證明交易預測力。

6461 益得 2026/06–08 單月合計為 NT$12.027M，去年同三月為 NT$71.644M，合計年增 -83.21%；8 月單月 +1102.98%、1–8 月累計 -56.20%。公司原註明 8 月增長與經銷商備貨有關，累計下降受去年 NT$68.04M 技術服務收入影響。三月彙總仍含該比較基期，故需同時呈現公司說明，不單獨作空方或多方判斷。


In [ ]:
import sys, json, collections, datetime
sys.path.insert(0, "/opt/airflow/dags")
from source_extension.registry import _open_connection

conn, cur, engine = _open_connection()
periods = (datetime.date(2026, 6, 1), datetime.date(2026, 7, 1), datetime.date(2026, 8, 1))
cur.execute("""
SELECT r.id, r.entity_id, r.source_key, r.effective_date, r.metric_value,
       MAX(CASE WHEN f.target_field='revenue_ytd_pct' THEN f.value_numeric END) AS ytd,
       r.payload
FROM source_records r
LEFT JOIN source_record_fields f ON f.record_hash=r.record_hash
WHERE r.source_key IN ('twse_monthly_revenue', 'tpex_monthly_revenue')
  AND r.event_type='monthly_revenue' AND r.quality_status='ok'
  AND r.effective_date IN ('2026-06-01', '2026-07-01', '2026-08-01')
GROUP BY r.id
ORDER BY r.source_key, r.entity_id, r.effective_date, r.event_time, r.id
""")
raw = cur.fetchall()
by_market_period = collections.defaultdict(list)
by_security = collections.defaultdict(dict)
source_rows = []
for row_id, code, source, period, yoy, ytd, payload in raw:
    obj = json.loads(payload) if isinstance(payload, str) else payload
    record = {
        "id": row_id,
        "current": float(obj["營業收入-當月營收"]),
        "prior": float(obj["營業收入-去年當月營收"]),
        "yoy": float(yoy) if yoy is not None else None,
        "ytd": float(ytd) if ytd is not None else None,
    }
    code = str(code)
    by_market_period[(source, period)].append(code)
    by_security[(source, code)][period.month] = record
    source_rows.append((source, code, period, record))

print("Accepted source rows and distinct codes by market/month:")
for source, period in sorted(by_market_period, key=lambda x:(x[0],x[1])):
    codes = by_market_period[(source, period)]
    print(source, period, "rows=", len(codes), "distinct=", len(set(codes)), "duplicate_grain_rows=", len(codes)-len(set(codes)))

print("\nMonthly YoY vs YTD YoY disagreement:")
for source, period in sorted(by_market_period, key=lambda x:(x[1],x[0])):
    records = [r for s,c,p,r in source_rows if s==source and p==period and r["yoy"] is not None and r["ytd"] is not None]
    conflicts = [r for r in records if r["yoy"] * r["ytd"] < 0]
    print(source, period, "paired=",len(records), "opposite=",len(conflicts), "pct=",round(100*len(conflicts)/len(records),2) if records else None)

three_month = {}
for (source, code), month_map in by_security.items():
    if not all(month in month_map for month in (6,7,8)):
        continue
    current_total = sum(month_map[m]["current"] for m in (6,7,8))
    prior_total = sum(month_map[m]["prior"] for m in (6,7,8))
    if prior_total <= 0:
        continue
    yoy_3m = (current_total / prior_total - 1) * 100
    latest = month_map[8]
    three_month[(source, code)] = {
        "current_total": current_total,
        "prior_total": prior_total,
        "yoy_3m": yoy_3m,
        "monthly_yoy": latest["yoy"],
        "ytd_yoy": latest["ytd"],
    }

print("\nThree-month coverage and sign conflicts:")
for source in ("twse_monthly_revenue", "tpex_monthly_revenue"):
    rows = [v for (s,_),v in three_month.items() if s==source]
    monthly_conflicts = sum(v["monthly_yoy"] is not None and v["monthly_yoy"]*v["yoy_3m"] < 0 for v in rows)
    ytd_conflicts = sum(v["ytd_yoy"] is not None and v["ytd_yoy"]*v["yoy_3m"] < 0 for v in rows)
    print(source, "complete_3m=",len(rows), "monthly_vs_3m=",monthly_conflicts,
          "pct=",round(100*monthly_conflicts/len(rows),2),
          "ytd_vs_3m=",ytd_conflicts, "pct=",round(100*ytd_conflicts/len(rows),2))

print("\n6461 益得:", three_month.get(("tpex_monthly_revenue", "6461")))
run_key = "2ef9378a635d3c04d0ffaa53ef098a20f8a2a124772e829bb4bec6769476e1f3"
cur.execute("SELECT entity_id, market, details_text FROM market_signal_candidates WHERE run_key=%s", (run_key,))
candidates = cur.fetchall()
matched = []
for code, market, details_text in candidates:
    source = "twse_monthly_revenue" if market == "上市" else "tpex_monthly_revenue"
    value = three_month.get((source, str(code)))
    if value:
        matched.append(value)
print("Oct 2 candidates with 3M data:", len(matched), "of", len(candidates))
print("candidate monthly_vs_3m conflicts:", sum(v["monthly_yoy"] is not None and v["monthly_yoy"]*v["yoy_3m"] < 0 for v in matched))
print("candidate YTD_vs_3m conflicts:", sum(v["ytd_yoy"] is not None and v["ytd_yoy"]*v["yoy_3m"] < 0 for v in matched))
cur.close()
conn.close()
